Transformação de dados

In [1]:
import pandas as pd
from camara_deputados.ingestion.data_loader import DataLoader
from camara_deputados.extraction.write import DataWrite
from camara_deputados.transformer.transformer import DataTransformer



In [2]:
# instâncias

bronze = DataLoader('bronze')
salva= DataWrite()
transforma = DataTransformer()



In [25]:
dfs_bronze = bronze.carregar_parquets()


📂 Lendo dados da camada bronze: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/notebooks/../data/bronze

✅ deputado_detalhamento: 1000 linhas, 34 colunas
✅ deputado_frentes: 271053 linhas, 6 colunas
✅ deputado_lista: 1000 linhas, 10 colunas
✅ frentes: 100 linhas, 5 colunas
✅ frentes_detalhamento: 100 linhas, 22 colunas
✅ partidos: 15 linhas, 5 colunas
✅ partidos_detalhamento: 15 linhas, 23 colunas
✅ proposicoes: 135 linhas, 10 colunas
✅ proposicoes_autores: 343 linhas, 8 colunas
✅ proposicoes_detalhamento: 135 linhas, 36 colunas
✅ proposicoes_temas: 141 linhas, 5 colunas
✅ proposicoes_votacoes: 819 linhas, 13 colunas
✅ tipos_Autor: 58 linhas, 5 colunas
✅ tipos_proposicao: 544 linhas, 5 colunas
✅ votacoes_detalhamento: 819 linhas, 21 colunas
✅ votacoes_orientacao: 1883 linhas, 7 colunas
✅ votos_deputados: 41678 linhas, 13 colunas

🎯 Carregamento finalizado.


# Cria camada silver

## Deputado

In [51]:
# criando a silver Deputado

df_deputadoDetalhamento = dfs_bronze['deputado_detalhamento']

colunas_origem = list(df_deputadoDetalhamento)

In [53]:
#renomeado as colunas 

mapeamento_deputados = {
    'id':('id_Mandato','int')
    ,'nomeCivil':('nom_NomeCivil','str')
    ,'sexo':('nom_Sexo','str')
    ,'dataNascimento':('dat_DataNasc','date')
    ,'dataFalecimento':('dat_DataFalecimento','date')
    ,'ufNascimento':('nom_UFNasc','str')
    ,'municipioNascimento':('nom_MunicipioNasci','str')
    ,'escolaridade':('nom_Escolaridade','str')
    ,'ultimoStatus.siglaPartido':( 'nom_SiglaPartido','str')
    ,'ultimoStatus.siglaUf':('nom_UFRepresenta', 'str')
    ,'ultimoStatus.idLegislatura':('id_legislatura','int')
    ,'ultimoStatus.email':('nom_Email','str')
    ,'ultimoStatus.nomeEleitoral':('nom_NomeEleitoral','str')
    ,'ultimoStatus.situacao':('nom_Situacao','str')
    ,'ultimoStatus.condicaoEleitoral':('nom_CondEleitoral','str')
    ,'source_id': ('id_Deputado', 'int')

}

df_s_deputado = transforma.rename_and_cast(df_s_deputado,mapeamento_deputados)

In [55]:
salva.save_parquet(df_s_deputado, 'silver_deputado', 'silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_deputado/silver_deputado.parquet


## Deputados Frentes

In [57]:
df_deputadoFrentes = dfs_bronze['deputado_frentes']

In [58]:
print(df_deputadoFrentes.columns)

Index(['id', 'uri', 'titulo', 'idLegislatura', 'source_id', 'data_extracao'], dtype='str')


In [60]:
mapeamento_frentes = {
    'id': ('id_Frente', 'int'),
    #'uri': ('des_Uri', 'str'),
    'titulo': ('nom_Titulo', 'str'),
    'idLegislatura': ('id_Legislatura', 'int'),
    'source_id': ('id_Deputado', 'int'),
    #'data_extracao': ('dat_DataExtracao', 'datetime')
}

df_s_frenteDeputado = transforma.rename_and_cast(df_deputadoFrentes,mapeamento_frentes)

In [61]:
df_s_frenteDeputado.sample(5)

,id_Frente,nom_Titulo,id_Legislatura,id_Deputado
217718,54432,Frente Parlamentar Mista em Defesa da Advocaci...,57,160758
247883,54294,Frente Parlamentar Mista de Portos e Aeroportos,57,198783
46681,53952,Frente Parlamentar em Defesa da Polícia Federal,56,178881
18325,53995,Frente Parlamentar em Defesa do BANCO DO NORDE...,56,178836
54373,54080,Frente Parlamentar Mista da Mineração,56,74471


In [62]:
salva.save_parquet(df_s_frenteDeputado,'silver_frenteDeputado', 'silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_frenteDeputado/silver_frenteDeputado.parquet


## Proposicões Detalhada

In [63]:
#criando a tabela silver das proposições

df_proposicaoDetalhamento = dfs_bronze['proposicoes_detalhamento']

colunas_origem = list(df_proposicaoDetalhamento.columns)
print(colunas_origem)

['id', 'uri', 'siglaTipo', 'codTipo', 'numero', 'ano', 'ementa', 'dataApresentacao', 'uriOrgaoNumerador', 'uriAutores', 'descricaoTipo', 'ementaDetalhada', 'keywords', 'uriPropPrincipal', 'uriPropAnterior', 'uriPropPosterior', 'urlInteiroTeor', 'urnFinal', 'texto', 'justificativa', 'statusProposicao.dataHora', 'statusProposicao.sequencia', 'statusProposicao.siglaOrgao', 'statusProposicao.uriOrgao', 'statusProposicao.uriUltimoRelator', 'statusProposicao.regime', 'statusProposicao.descricaoTramitacao', 'statusProposicao.codTipoTramitacao', 'statusProposicao.descricaoSituacao', 'statusProposicao.codSituacao', 'statusProposicao.despacho', 'statusProposicao.url', 'statusProposicao.ambito', 'statusProposicao.apreciacao', 'source_id', 'data_extracao']


In [64]:
# as colunas sinalizadas com # não subirão para a silver

mapeamento_proposicao= {
    'id':('id_Proposicao','int')
    #,'uri'
    ,'siglaTipo':('nom_TipoProposicao','str')
    ,'codTipo':('cod_Tipo','int')
    ,'numero':('num_NumeroProp','int')
    ,'ano':('num_Ano', 'int')
    ,'ementa':('nom_Ementa','str')
    ,'dataApresentacao':('dat_Apresentacao','date')
    #,'uriOrgaoNumerador'
    ,'uriAutores':('uri_Autor','str')
    #,'descricaoTipo'
    #,'ementaDetalhada'
    ,'keywords':('nom_Keywords','str')
    #,'uriPropPrincipal'
    #,'uriPropAnterior'
    #,'uriPropPosterior'
    #,'urlInteiroTeor'
    #,'urnFinal'
    #,'texto'
    #,'justificativa'
    #,'statusProposicao.dataHora'
    #,'statusProposicao.sequencia'
    ,'statusProposicao.siglaOrgao':('nom_SiglaOrgao', 'str')
    #,'statusProposicao.uriOrgao'
    ,'statusProposicao.uriUltimoRelator':('uri_Relator', 'str')
    ,'statusProposicao.regime':('nom_regime','str')
    ,'statusProposicao.descricaoTramitacao':('nom_TipoTramitacao','str')
    ,'statusProposicao.codTipoTramitacao':('cod_TipoTramitacao','int')
    ,'statusProposicao.descricaoSituacao':('cod_TipoSiituacao','str')
    ,'statusProposicao.codSituacao':('cod_Situacao','int')
    #,'statusProposicao.despacho'
    ,'statusProposicao.url':('nom_LinkProposicao','str')
    #,'statusProposicao.ambito'
    #,'statusProposicao.apreciacao'
    #,'source_id'
    #,'data_extracao'
}

df_s_proposicao = transforma.rename_and_cast(df_proposicaoDetalhamento, mapeamento_proposicao)

In [65]:
# Pega o id da uri

colunas_uri = {'uri_Autor':('nom_TipoAutor','id_Autor'), 'uri_Relator':('nom_TipoRelator','id_Relator')}

df_s_proposicao = transforma.extrair_tipos_e_ids(df_s_proposicao, colunas_uri)



In [66]:
df_s_proposicao.info()

<class 'pandas.DataFrame'>
RangeIndex: 135 entries, 0 to 134
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_Proposicao       135 non-null    Int64         
 1   nom_TipoProposicao  135 non-null    string        
 2   cod_Tipo            135 non-null    Int64         
 3   num_NumeroProp      135 non-null    Int64         
 4   num_Ano             135 non-null    Int64         
 5   nom_Ementa          135 non-null    string        
 6   dat_Apresentacao    135 non-null    datetime64[us]
 7   uri_Autor           135 non-null    string        
 8   nom_Keywords        105 non-null    string        
 9   nom_SiglaOrgao      133 non-null    string        
 10  uri_Relator         85 non-null     string        
 11  nom_regime          135 non-null    string        
 12  nom_TipoTramitacao  133 non-null    string        
 13  cod_TipoTramitacao  133 non-null    Int64         
 14  cod_T

In [67]:
# salva as proposições na silver

salva.save_parquet(df_s_proposicao, 'silver_proposicao','silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_proposicao/silver_proposicao.parquet


## Autores Proposicões

In [68]:
df_proposicaoAutores = dfs_bronze['proposicoes_autores']

mapeamento_autores = list(df_proposicaoAutores.columns)

In [69]:
mapeamento_autores = {
    'uri': ('uri_Autor', 'str'),
    'nome': ('nom_Autor', 'str'),
    'codTipo': ('cod_TipoAutor', 'int'),
    'tipo': ('nom_TipoAutor', 'str'),
    'ordemAssinatura': ('num_OrdemAssinatura', 'int'),
    'proponente': ('ind_Proponente', 'str'),  
    'source_id': ('id_Proposicao', 'int'),
    'data_extracao': ('dat_Extracao', 'date')
}

df_s_autores  = transforma.rename_and_cast(df_proposicaoAutores, mapeamento_autores)

## Resgata ID da uri

df_s_autores = transforma.extrair_tipos_e_ids(
    df_s_autores,
    {
        'uri_Autor': ('nom_TipoAutor', 'id_Autor')
    }
)

In [70]:
df_s_autores.columns

Index(['uri_Autor', 'nom_Autor', 'cod_TipoAutor', 'nom_TipoAutor',
       'num_OrdemAssinatura', 'ind_Proponente', 'id_Proposicao',
       'dat_Extracao', 'id_Autor'],
      dtype='str')

In [22]:
df_s_autores.sample(20)

,uri_Autor,nom_Autor,cod_TipoAutor,nom_TipoAutor,num_OrdemAssinatura,ind_Proponente,id_proposicao,dat_Extracao,id_Autor
7,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2345487,2026-04-19 18:19:55.573900,253
4,https://dadosabertos.camara.leg.br/api/v2/depu...,Mara Gabrilli,10000,deputados,1,1,2074843,2026-04-19 18:19:55.573900,160565
26,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2416825,2026-04-19 18:19:55.573900,253
31,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2481877,2026-04-19 18:19:55.573900,253
12,https://dadosabertos.camara.leg.br/api/v2/orga...,Senado Federal - Roberto Muniz,40000,orgaos,1,1,2345498,2026-04-19 18:19:55.573900,78
17,https://dadosabertos.camara.leg.br/api/v2/depu...,Sandro Alex,10000,deputados,1,1,2092056,2026-04-19 18:19:55.573900,160621
6,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2345485,2026-04-19 18:19:55.573900,253
9,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2345494,2026-04-19 18:19:55.573900,253
41,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2481889,2026-04-19 18:19:55.573900,253
50,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2599894,2026-04-19 18:19:55.573900,253


In [72]:
salva.save_parquet(df_s_autores, 'silver_autores','silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_autores/silver_autores.parquet


## Silver temas das proposicoes

In [73]:
df_proposicoesTemas = dfs_bronze['proposicoes_temas']

list_temas = list(df_proposicoesTemas.columns)

mapeamento_temas = {
    'codTema': ('cod_Tema', 'int'),
    'tema': ('nom_Tema', 'str'),
    'relevancia': ('num_Relevancia', 'int'),
    'source_id': ('id_proposicao', 'int'),
    'data_extracao': ('dat_Extracao', 'date')
}

df_s_temasProposicoes = transforma.rename_and_cast(df_proposicoesTemas,mapeamento_temas)

In [74]:
salva.save_parquet(df_s_temasProposicoes, 'silver_temasProposicao', 'silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_temasProposicao/silver_temasProposicao.parquet


## Proposições votações

In [75]:
df_proposicoesVotacoes = dfs_bronze['proposicoes_votacoes']

In [76]:
print(list(df_proposicoesVotacoes.columns))

['id', 'uri', 'data', 'dataHoraRegistro', 'siglaOrgao', 'uriOrgao', 'uriEvento', 'proposicaoObjeto', 'uriProposicaoObjeto', 'descricao', 'aprovacao', 'source_id', 'data_extracao']


In [83]:
mapeamento_votacao = {
    'id': ('id_Votacao', 'str'),
    #'uri': ('uri_Votacao', 'str'),
    'data': ('dat_DataVotacao', 'date'),
    'dataHoraRegistro': ('dat_DataRegistro', 'date'),
    #'siglaOrgao': ('nom_SiglaOrgao', 'str'),
    'uriOrgao': ('uri_Orgao', 'str'),
    #'uriVotacao': ('uri_votacaoDetalhe', 'str'),
    #'proposicaoObjeto': ('nom_ProposicaoObjeto', 'str'),
    #'uriProposicaoObjeto': ('uri_Proposicao', 'str'),
    'descricao': ('nom_Descricao', 'str'),
    'aprovacao': ('ind_Aprovado', 'str'),  
    'source_id': ('id_Proposicao', 'int'),
    #'data_extracao': ('dat_Extracao', 'date')
}

df_s_votacaoProposicao = transforma.rename_and_cast(df_proposicoesVotacoes,mapeamento_votacao)

In [84]:
df_s_votacaoProposicao = transforma.extrair_ids(
    df_s_votacaoProposicao,
    {'uri_Orgao': 'id_Orgao'}
)

In [85]:
salva.save_parquet(df_s_votacaoProposicao, 'silver_votacao_proposicao','silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_votacao_proposicao/silver_votacao_proposicao.parquet


## Votacões

In [4]:
df_votacaoDetalhamento = dfs_bronze['votacoes_detalhamento']

In [8]:
df_votacaoDetalhamento['proposicoesAfetadas'].iloc[4]

array([{'ano': 2018, 'codTipo': 291, 'dataApresentacao': '2018-01-12T09:28', 'ementa': 'Altera a Lei nº 13.089, de 12 de janeiro de 2015, que institui o Estatuto da Metrópole, e a Lei nº 12.587, de 3 de janeiro de 2012, que institui as diretrizes da Política Nacional de Mobilidade Urbana.  NOVA EMENTA: Altera as Leis nºs 13.089, de 12 de janeiro de 2015 (Estatuto da Metrópole), e 12.587, de 3 de janeiro de 2012, que institui as diretrizes da Política Nacional de Mobilidade Urbana.', 'id': 2167559, 'numero': 818, 'siglaTipo': 'MPV', 'uri': 'https://dadosabertos.camara.leg.br/api/v2/proposicoes/2167559'}],
      dtype=object)

In [9]:
mapeamento_votacaoDetalhamento = {
    'id': ('id_Votacao', 'str'),
    #'uri': ('des_UriVotacao', 'str'),
    'data': ('dat_DataVotacao', 'date'),
    'dataHoraRegistro': ('dat_DataHoraRegistro', 'date'),
    
    'siglaOrgao': ('nom_SiglaOrgao', 'str'),
    #'uriOrgao': ('des_UriOrgao', 'str'),
    'idOrgao': ('id_Orgao', 'int'),
    
    #'uriEvento': ('des_UriEvento', 'str'),
    'idEvento': ('id_Evento', 'int'),
    
    'descricao': ('des_DescricaoVotacao', 'str'),
    'aprovacao': ('ind_Aprovacao', 'str'),
    
    'descUltimaAberturaVotacao': ('des_UltimaAbertura', 'str'),
    'dataHoraUltimaAberturaVotacao': ('dat_UltimaAbertura', 'date'),
    
    'efeitosRegistrados': ('des_Efeitos', 'str'),
    'objetosPossiveis': ('des_Objetos', 'str'),
    #'proposicoesAfetadas': ('des_Proposicoes', 'str'),
    
    'ultimaApresentacaoProposicao.dataHoraRegistro': ('dat_UltimaApresentacao', 'date'),
    'ultimaApresentacaoProposicao.descricao': ('des_UltimaApresentacaoDesc', 'str'),
    'ultimaApresentacaoProposicao.uriProposicaoCitada': ('des_UriProposicao', 'str'),
    
    #'source_id': ('id_Votacao', 'str'),
    #'data_extracao': ('dat_DataExtracao', 'date')
}

In [21]:
df_s_votacoesDetalhamento = transforma.rename_and_cast(df_votacaoDetalhamento,mapeamento_votacaoDetalhamento)

In [14]:
df_proposicoesAfetadas = transforma.explode_and_extract(
    df_votacaoDetalhamento,
    'proposicoesAfetadas',
     keys=['id', 'siglaTipo', 'numero', 'ano'],
    rename_map={
        'id': 'id_Proposicao',
        'siglaTipo': 'nom_SiglaTipoProposicao',
        'numero': 'num_Proposicao',
        'ano': 'num_AnoProposicao'
    }
)

In [22]:
salva.save_parquet(df_proposicoesAfetadas, 'silver_votacao_detalhamento','silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_votacao_detalhamento/silver_votacao_detalhamento.parquet


In [20]:
df_proposicoesAfetadas.describe()

,idOrgao,idEvento,aprovacao,data_extracao
count,819.000000,732.000000,797.000000,819
mean,176610.846154,54615.610656,0.894605,2026-05-01 14:34:11.672524
min,4.000000,7837.000000,0.000000,2026-05-01 14:34:11.672524
25%,180.000000,48754.000000,1.000000,2026-05-01 14:34:11.672524
50%,2014.000000,57143.000000,1.000000,2026-05-01 14:34:11.672524
75%,537650.000000,66065.250000,1.000000,2026-05-01 14:34:11.672524
max,539387.000000,81692.000000,1.000000,2026-05-01 14:34:11.672524
std,245133.535531,15290.466718,0.307255,NaN


## Votações Orientação

In [44]:
df_votacoesOrientacao = dfs_bronze['votacoes_orientacao']

In [45]:
df_votacoesOrientacao.head()

,orientacaoVoto,codTipoLideranca,siglaPartidoBloco,codPartidoBloco,uriPartidoBloco,source_id,data_extracao
0,Sim,P,NOVO,37901.0,https://dadosabertos.camara.leg.br/api/v2/part...,559138-241,2026-05-01 14:46:37.688038
1,,P,PSB,36832.0,https://dadosabertos.camara.leg.br/api/v2/part...,559138-241,2026-05-01 14:46:37.688038
2,Sim,B,Oposição,NaN,NaN,559138-241,2026-05-01 14:46:37.688038
3,Não,B,Governo,NaN,NaN,559138-241,2026-05-01 14:46:37.688038
4,Sim,P,PL,37906.0,https://dadosabertos.camara.leg.br/api/v2/part...,559138-241,2026-05-01 14:46:37.688038


In [46]:
mapeamento_orientacaoVoto = {
    'orientacaoVoto': ('nom_OrientacaoVoto', 'str'),
    'codTipoLideranca': ('id_TipoLideranca', 'int'),
    
    'siglaPartidoBloco': ('nom_SiglaPartidoBloco', 'str'),
    'codPartidoBloco': ('id_PartidoBloco', 'int'),
    'uriPartidoBloco': ('des_UriPartidoBloco', 'str'),
    
    'source_id': ('id_Votacao', 'str'),
    'data_extracao': ('dat_DataExtracao', 'date')
}

df_s_votosOrientacao = transforma.rename_and_cast(df_votacoesOrientacao,
                                                mapeamento_orientacaoVoto)

In [48]:
salva.save_parquet(
    df_s_votosOrientacao,
    'silver_orientacao',
    'silver'
)

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_orientacao/silver_orientacao.parquet


In [ ]:
df_tiposAutor = dfs_bronze['tipos_Autor']

In [ ]:
df_tiposAutor.columns

Index(['cod', 'sigla', 'nome', 'descricao', 'data_extracao'], dtype='str')

In [ ]:
maping_autor = {'cod':('id_Autor','int'),
 #'sigla',
 'nome':('nom_NomeAutor','str'),
 #'descricao',
 #'data_extracao'
}



s_tiposAutores = transforma.rename_and_cast(
    df_tiposAutor,
    maping_autor,
)

In [ ]:
salva.save_parquet(
    s_tiposAutores,
    'silver_tipos_autores',
    'silver'
)

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_tipos_autores/silver_tipos_autores.parquet


In [26]:
df_votos = dfs_bronze['votos_deputados']

In [27]:
df_votos.columns

Index(['tipoVoto', 'dataRegistroVoto', 'deputado_.id', 'deputado_.uri',
       'deputado_.nome', 'deputado_.siglaPartido', 'deputado_.uriPartido',
       'deputado_.siglaUf', 'deputado_.idLegislatura', 'deputado_.urlFoto',
       'deputado_.email', 'source_id', 'data_extracao'],
      dtype='str')

In [29]:
df_votos.head()

,tipoVoto,dataRegistroVoto,deputado_.id,deputado_.uri,deputado_.nome,deputado_.siglaPartido,deputado_.uriPartido,deputado_.siglaUf,deputado_.idLegislatura,deputado_.urlFoto,deputado_.email,source_id,data_extracao
0,Sim,2024-11-27T22:23:36,220571,https://dadosabertos.camara.leg.br/api/v2/depu...,Daniel Agrobom,PL,https://dadosabertos.camara.leg.br/api/v2/part...,GO,57,https://www.camara.leg.br/internet/deputado/ba...,dep.danielagrobom@camara.leg.br,559138-241,2026-05-01 18:15:23.468968
1,Não,2024-11-27T22:23:30,163321,https://dadosabertos.camara.leg.br/api/v2/depu...,Ronaldo Nogueira,REPUBLICANOS,https://dadosabertos.camara.leg.br/api/v2/part...,RS,57,https://www.camara.leg.br/internet/deputado/ba...,NaN,559138-241,2026-05-01 18:15:23.468968
2,Não,2024-11-27T22:23:10,220645,https://dadosabertos.camara.leg.br/api/v2/depu...,Erika Hilton,PSOL,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.erikahilton@camara.leg.br,559138-241,2026-05-01 18:15:23.468968
3,Sim,2024-11-27T22:23:03,204472,https://dadosabertos.camara.leg.br/api/v2/depu...,José Medeiros,PL,https://dadosabertos.camara.leg.br/api/v2/part...,MT,57,https://www.camara.leg.br/internet/deputado/ba...,dep.josemedeiros@camara.leg.br,559138-241,2026-05-01 18:15:23.468968
4,Sim,2024-11-27T22:23:03,220655,https://dadosabertos.camara.leg.br/api/v2/depu...,Mario Frias,PL,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.mariofrias@camara.leg.br,559138-241,2026-05-01 18:15:23.468968


In [30]:
mapping_votos = {'tipoVoto':('nom_Voto','str')
                ,'dataRegistroVoto':('dat_DataRegistro','date')
                ,'deputado_.id':('id_Deputado','int')
                #,'deputado_.uri'
                #,'deputado_.nome'
                #,'deputado_.siglaPartido'
                #,'deputado_.uriPartido'
                #,'deputado_.siglaUf'
                #,'deputado_.idLegislatura'
                #,'deputado_.urlFoto'
                #,'deputado_.email'
                ,'source_id':('id_Votacao','str')
                #,'data_extracao'

}

s_votos_deputados = transforma.rename_and_cast(
    df_votos,
    mapping_votos,
    
)

In [32]:
salva.save_parquet(s_votos_deputados,
                   'silver_votos_deputados',
                   'silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_votos_deputados/silver_votos_deputados.parquet


In [34]:
df_votacoes_orientacao = dfs_bronze['votacoes_orientacao']

df_votacoes_orientacao.columns

Index(['orientacaoVoto', 'codTipoLideranca', 'siglaPartidoBloco',
       'codPartidoBloco', 'uriPartidoBloco', 'source_id', 'data_extracao'],
      dtype='str')

In [38]:
mapping_orientacao = {'orientacaoVoto':('nom_Voto','str')
#,'codTipoLideranca'
,'siglaPartidoBloco':('nom_SiglaPartidoBloco','str')
#,'codPartidoBloco'
,#'uriPartidoBloco'
'source_id':('id_Votacao','str')
}

In [40]:
s_votacao = transforma.rename_and_cast(
    df_votacoes_orientacao,
    mapping_orientacao,
)

salva.save_parquet(
    s_votacao,
    'silver_orientacao',
    'silver'
)

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_orientacao/silver_orientacao.parquet


In [43]:
s_votacao.info()

<class 'pandas.DataFrame'>
RangeIndex: 1883 entries, 0 to 1882
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   nom_Voto               1883 non-null   string        
 1   nom_SiglaPartidoBloco  1883 non-null   string        
 2   id_Votacao             1883 non-null   string        
 3   data_extracao          1883 non-null   datetime64[us]
dtypes: datetime64[us](1), string(3)
memory usage: 96.3 KB
